# 07 — Definitional Patterns: what do LLMs encode about "Digital Humanities"?

The earlier notebooks (02–06) examine *what the models produce* through translations, review signals, prompt comparisons, embeddings, and rationale style. This notebook turns to a different question: **what do the models think Digital Humanities *is*?** The rationale column gives us an indirect window into each model's definition of the term, and variation across models, variants, and target languages shows how unevenly they frame a concept that DH scholars themselves continue to debate.

Four motivating questions structure the analysis:

1. **Inductive patterns.** What recurrent vocabularies emerge before I impose a hand-built taxonomy, and how stable are they across topic counts and clustering choices?
2. **Definition versus authorization.** Can I distinguish language that describes DH from language that makes a proposed translation appear legitimate?
3. **Model and prompt variation.** How do those overlapping signals vary across services and prompt conditions?
4. **Component asymmetry.** Do models gloss the technical and humanities components of the phrase with comparable specificity?

**Method**: I first extract likely definitional spans and compare NMF topic discovery with a BERTopic-style embedding pipeline. I then translate the most stable vocabularies into transparent, overlapping rules and test their rhetorical coherence with grouped, trigger-masked classification. The original four-register rules remain as a deductive benchmark; component-level pattern matching and review-signal comparisons provide secondary checks. None of these methods validates the translations or turns the discovered patterns into exclusive semantic classes.

**Input**: `evaluation/disagreement_explorer_data.csv`, which enforces translation/rationale pairing, plus the Tier 2 decisions in `manual_exclusions.csv`.  
**Scope**: English-register variants only — `minimal`, `github_searcher`, `judge`. The `fluent_speaker` variant produces target-language rationales and is excluded from this analysis.

In [1]:
import os, sys, re, warnings
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import altair as alt

warnings.filterwarnings("ignore", message="n_jobs value 1 overridden to 1 by setting random_state.*")

alt.data_transformers.enable('vegafusion')

PROJECT_ROOT = Path('..')
if not (PROJECT_ROOT / 'scripts' / 'utils.py').exists():
    PROJECT_ROOT = Path('.')
sys.path.insert(0, str(PROJECT_ROOT))
from scripts.utils import get_language_family, read_csv_file, save_chart

DATA_DIR = PROJECT_ROOT / "datasets"
TERM      = 'Digital Humanities'
TERM_SLUG = TERM.lower().replace(' ', '_')
FIGURE_DIR = PROJECT_ROOT / 'docs' / 'papers' / 'translation-to-transmogrification' / 'figures' / 'analysis'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# English-rationale variants only — fluent_speaker uses target language
ENGLISH_VARIANTS = ['minimal', 'github_searcher', 'judge']
ALL_VARIANTS     = ['minimal', 'fluent_speaker', 'github_searcher', 'judge']

LLM_SERVICES = ['claude', 'openai', 'gemini', 'deepseek', 'llama', 'gemma', 'qwen', 'mistral']
FRONTIER     = {'claude', 'openai', 'gemini', 'deepseek'}
SMALL        = {'llama', 'gemma', 'qwen', 'mistral'}

EVAL_DIR = os.path.join(DATA_DIR, 'translated_terms', TERM_SLUG, 'evaluation')
EXPLORER_CSV = os.path.join(EVAL_DIR, 'disagreement_explorer_data.csv')
EXCL_PATH = os.path.join(EVAL_DIR, 'manual_exclusions.csv')


Retrieving translation pipeline data directory path...

Rationale matrix: ../datasets/translated_terms/digital_humanities/evaluation/disagreement_explorer_data.csv


## 7.1 — Loading the rationale corpus

Build a single long-format DataFrame with one row per (service, variant, language) rationale from the paired rationale matrix used in Notebook 06. Drop empty, short, or placeholder rationales (`'No rationale provided'`, `'N/A'`, etc.), apply Tier 2 manual translation/rationale exclusions, and restrict to English-register variants. The output is the corpus that all subsequent analyses operate on.

In [2]:
PLACEHOLDER_RAT = {
    'no rationale provided', 'no rationale', 'n/a', 'none',
    'not applicable', 'no explanation provided', 'no reason provided',
    '[rationale truncated — max_tokens]',
}

def valid_rat(value):
    if not isinstance(value, str) or len(value.strip()) <= 20:
        return False
    return value.strip().rstrip('.').lower() not in PLACEHOLDER_RAT

explorer_df = read_csv_file(EXPLORER_CSV)
exclusion_lookup = {}
if os.path.exists(EXCL_PATH):
    exclusions = read_csv_file(EXCL_PATH)
    for _, row in exclusions.iterrows():
        code = str(row.get('language_code', '')).strip()
        service = str(row.get('service', '')).strip().lower()
        if not code or not service or service == 'nan':
            continue
        exclusion_lookup[(code, service)] = {
            'translation': str(row.get('exclude_translation', 'False')).strip().lower() == 'true',
            **{
                f'rationale_{variant}': str(row.get(f'exclude_rationale_{variant}', 'False')).strip().lower() == 'true'
                for variant in ALL_VARIANTS
            },
        }

rows = []
for service in LLM_SERVICES:
    for variant in ENGLISH_VARIANTS:
        rationale_col = f'{service}_rationale_{variant}'
        if rationale_col not in explorer_df.columns:
            continue
        for _, row in explorer_df[['language_code', rationale_col]].iterrows():
            code = str(row.get('language_code', '')).strip()
            rationale = row.get(rationale_col)
            decisions = exclusion_lookup.get((code, service), {})
            if (decisions.get('translation', False)
                    or decisions.get(f'rationale_{variant}', False)
                    or not valid_rat(rationale)):
                continue
            rows.append({
                'service': service,
                'variant': variant,
                'language_code': code,
                'rationale': rationale.strip(),
                'model_size': 'frontier' if service in FRONTIER else 'small',
            })
rationale_df = pd.DataFrame(rows)
print(f'Total English-register rationales: {len(rationale_df)}')
print(f'  By service:\n{rationale_df["service"].value_counts().to_string()}')

Total English-register rationales: 20661
  By service:
service
claude      2642
qwen        2635
deepseek    2626
gemma       2616
mistral     2587
gemini      2563
openai      2547
llama       2445


## 7.2 — Inductive Discovery of Definitional Patterns

Before treating the four hand-built registers as settled categories, I use two unsupervised approaches to ask what patterns emerge from the rationales themselves. The first is non-negative matrix factorization (NMF) over TF-IDF features, which permits each rationale to combine several topics. The second is a BERTopic-style pipeline assembled from the project's existing dependencies: sentence-transformer embeddings, UMAP, HDBSCAN, and class-level TF-IDF summaries. I use the phrase *BERTopic-style* because the `bertopic` package itself is not installed.

The unit is one likely definitional sentence per rationale. A broad cue list gives priority to sentences that describe DH as a field, intersection, method, object of study, form of knowledge, or combination of concepts. When no sentence contains those cues, I retain the sentence with the strongest concentration of the source concept's component vocabulary. Before modeling, I remove service and prompt names, the target-language name and code, the proposed translated term, and non-ASCII strings likely to encode the target script. This reduces obvious metadata and script clustering without claiming to isolate pure meaning.


In [3]:
from scripts.analysis.rationale_classifier import (
    mask_language_name,
    mask_prompt_metadata,
    mask_translated_term,
)

DEFINITION_SENTENCE_SPLIT = re.compile(r"(?<=[.!?])\s+|\n+")
DEFINITION_STRONG_CUES = re.compile(
    r"\b(?:refers? to|represents?|concept of|interdisciplinary|academic field|academic discipline|"
    r"field of|intersection|study of|studies of|use of digital|digital (?:tools|methods|technology)|"
    r"human (?:culture|experience|society|knowledge)|knowledge|learning|"
    r"combines?.{0,80}(?:digital|humanities)|(?:digital|humanities).{0,80}combines?)\b",
    re.IGNORECASE,
)
DEFINITION_COMPONENT_CUES = re.compile(
    r"\b(?:digital humanities|digital|humanities|technology|computational|scholarship|"
    r"culture|knowledge|learning)\b",
    re.IGNORECASE,
)

def select_definitional_span(text):
    sentences = [
        re.sub(r"^[*\-•\s]+", "", sentence).strip()
        for sentence in DEFINITION_SENTENCE_SPLIT.split(text)
        if 25 <= len(sentence.strip()) <= 700
    ]
    if not sentences:
        return text[:700], False
    scored = []
    for position, sentence in enumerate(sentences):
        strong_hits = len(DEFINITION_STRONG_CUES.findall(sentence))
        component_hits = len(DEFINITION_COMPONENT_CUES.findall(sentence))
        scored.append((5 * strong_hits + component_hits, strong_hits, -position, sentence))
    _, strong_hits, _, selected = max(scored)
    return selected, strong_hits > 0

explorer_lookup = explorer_df.set_index("language_code")
topic_rows = []
for _, row in rationale_df.iterrows():
    span_raw, has_definition_cue = select_definitional_span(row["rationale"])
    source_row = explorer_lookup.loc[row["language_code"]]
    if isinstance(source_row, pd.DataFrame):
        source_row = source_row.iloc[0]
    language_name = str(source_row.get("language_name", ""))
    translation = source_row.get(f"{row['service']}_term_{row['variant']}", "")
    translation = "" if pd.isna(translation) else str(translation)

    span_clean = mask_prompt_metadata(span_raw)
    span_clean = mask_language_name(span_clean, language_name, row["language_code"])
    span_clean = mask_translated_term(span_clean, translation)
    span_clean = re.sub(r"[^\x00-\x7F]+", " ", span_clean)
    span_clean = re.sub(r"\s+", " ", span_clean).strip()
    if not span_clean:
        span_clean = re.sub(r"\s+", " ", span_raw).strip()

    topic_rows.append({
        "service": row["service"],
        "variant": row["variant"],
        "language_code": row["language_code"],
        "language_name": language_name,
        "span_raw": span_raw,
        "span_clean": span_clean,
        "has_definition_cue": has_definition_cue,
    })

definitional_spans = pd.DataFrame(topic_rows)
coverage_summary = (
    definitional_spans.groupby(["service", "variant"])["has_definition_cue"]
    .agg(["sum", "count", "mean"])
    .reset_index()
)
coverage_summary["mean"] = (coverage_summary["mean"] * 100).round(1)

print(f"Definitional spans selected: {len(definitional_spans):,}")
print(f"Unique cleaned spans: {definitional_spans['span_clean'].nunique():,}")
print(
    "Rationales with at least one strong definitional cue: "
    f"{definitional_spans['has_definition_cue'].sum():,} "
    f"({definitional_spans['has_definition_cue'].mean() * 100:.1f}%)"
)
display(coverage_summary)


Definitional spans selected: 20,661
Unique cleaned spans: 18,533
Rationales with at least one strong definitional cue: 12,339 (59.7%)


,service,variant,sum,count,mean
0,claude,github_searcher,510,880,58.0
1,claude,judge,629,881,71.4
2,claude,minimal,556,881,63.1
3,deepseek,github_searcher,431,875,49.3
4,deepseek,judge,581,881,65.9
5,deepseek,minimal,648,870,74.5
6,gemini,github_searcher,462,848,54.5
7,gemini,judge,825,869,94.9
8,gemini,minimal,580,846,68.6
9,gemma,github_searcher,561,870,64.5


### NMF Topic-Count and Seed Sensitivity

I fit NMF solutions from three through eight topics under three random seeds. Generic task words such as *translation*, *term*, *language*, and the source phrase are removed from the TF-IDF vocabulary so that they do not dominate every topic. Stability is the mean cosine similarity between topics after aligning each seeded solution to the first run with the Hungarian algorithm. Topic diversity is the share of unique terms among each solution's fifteen highest-weight terms. Neither statistic determines the correct number of concepts; together they show where additional topics remain reproducible rather than simply fragmenting the vocabulary.


In [4]:
from scipy.optimize import linear_sum_assignment
from sklearn.decomposition import NMF
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

TOPIC_TASK_STOPWORDS = {
    "translation", "translate", "translated", "translating", "term", "phrase", "word",
    "language", "english", "github", "search", "candidate", "proposed", "appropriate",
    "accurate", "likely", "means", "meaning", "digital", "humanities", "dh",
}
topic_stopwords = sorted(set(ENGLISH_STOP_WORDS) | TOPIC_TASK_STOPWORDS)

topic_vectorizer = TfidfVectorizer(
    stop_words=topic_stopwords,
    ngram_range=(1, 2),
    min_df=10,
    max_df=0.90,
    max_features=15_000,
    sublinear_tf=True,
)
topic_tfidf = topic_vectorizer.fit_transform(definitional_spans["span_clean"])
topic_terms = np.asarray(topic_vectorizer.get_feature_names_out())

nmf_models = {}
nmf_weights = {}
nmf_sensitivity_rows = []
for n_topics in range(3, 9):
    seed_models = []
    seed_weights = []
    for seed in [17, 42, 89]:
        model = NMF(
            n_components=n_topics,
            init="nndsvdar",
            random_state=seed,
            max_iter=500,
            l1_ratio=0.1,
        )
        weights = model.fit_transform(topic_tfidf)
        seed_models.append(model)
        seed_weights.append(weights)

    reference = seed_models[0].components_
    aligned_scores = []
    for model in seed_models[1:]:
        similarity = cosine_similarity(reference, model.components_)
        row_ind, col_ind = linear_sum_assignment(-similarity)
        aligned_scores.extend(similarity[row_ind, col_ind].tolist())

    top_term_ids = [
        component.argsort()[-15:][::-1]
        for component in reference
    ]
    diversity = len(set(np.concatenate(top_term_ids))) / (n_topics * 15)
    assignments = seed_weights[0].argmax(axis=1)
    shares = np.bincount(assignments, minlength=n_topics) / len(assignments)

    nmf_sensitivity_rows.append({
        "n_topics": n_topics,
        "mean_aligned_topic_similarity": float(np.mean(aligned_scores)),
        "topic_diversity": float(diversity),
        "smallest_topic_share": float(shares.min()),
        "reconstruction_error": float(seed_models[0].reconstruction_err_),
    })
    nmf_models[n_topics] = seed_models
    nmf_weights[n_topics] = seed_weights

nmf_sensitivity = pd.DataFrame(nmf_sensitivity_rows)
display(nmf_sensitivity.round(3))

nmf_sensitivity_long = nmf_sensitivity.melt(
    id_vars="n_topics",
    value_vars=["mean_aligned_topic_similarity", "topic_diversity"],
    var_name="metric",
    value_name="value",
)
nmf_sensitivity_long["metric"] = nmf_sensitivity_long["metric"].map({
    "mean_aligned_topic_similarity": "Seed stability",
    "topic_diversity": "Top-term diversity",
})
nmf_sensitivity_chart = alt.Chart(nmf_sensitivity_long).mark_line(
    point=alt.OverlayMarkDef(filled=True, size=70),
    strokeWidth=3,
).encode(
    x=alt.X("n_topics:O", title="Number of NMF topics"),
    y=alt.Y("value:Q", title="Score", scale=alt.Scale(domain=[0, 1])),
    color=alt.Color(
        "metric:N",
        title=None,
        scale=alt.Scale(
            domain=["Seed stability", "Top-term diversity"],
            range=["#2A9D8F", "#E76F51"],
        ),
    ),
    tooltip=["n_topics:O", "metric:N", alt.Tooltip("value:Q", format=".3f")],
).properties(
    width=500,
    height=280,
    title=alt.TitleParams(
        "NMF Sensitivity Across Topic Counts",
        subtitle="Topic alignment compares three random seeds at each solution size.",
        anchor="start",
    ),
).configure_view(stroke=None).configure_legend(orient="top")

save_chart(
    nmf_sensitivity_chart,
    str(FIGURE_DIR / "definitional_nmf_topic_sensitivity.png"),
    scale_factor=2.0,
)
display(nmf_sensitivity_chart)

# Six topics provide a deliberately granular candidate solution for inspection.
NMF_CANDIDATE_K = 6
candidate_model = nmf_models[NMF_CANDIDATE_K][1]  # seed 42
candidate_weights = nmf_weights[NMF_CANDIDATE_K][1]
candidate_assignments = candidate_weights.argmax(axis=1)
definitional_spans["nmf_topic"] = candidate_assignments

nmf_topic_rows = []
for topic_id, component in enumerate(candidate_model.components_):
    top_ids = component.argsort()[-12:][::-1]
    topic_indices = np.where(candidate_assignments == topic_id)[0]
    exemplar_indices = topic_indices[
        np.argsort(candidate_weights[topic_indices, topic_id])[-3:][::-1]
    ]
    service_shares = definitional_spans.iloc[topic_indices]["service"].value_counts(normalize=True)
    prompt_shares = definitional_spans.iloc[topic_indices]["variant"].value_counts(normalize=True)
    nmf_topic_rows.append({
        "topic": f"NMF {topic_id + 1}",
        "n_spans": len(topic_indices),
        "pct_spans": len(topic_indices) / len(definitional_spans) * 100,
        "top_terms": "; ".join(topic_terms[top_ids]),
        "representative_excerpt": definitional_spans.iloc[exemplar_indices[0]]["span_raw"],
        "largest_service_share": float(service_shares.iloc[0]),
        "largest_service": service_shares.index[0],
        "largest_prompt_share": float(prompt_shares.iloc[0]),
        "largest_prompt": prompt_shares.index[0],
    })

nmf_topic_summary = pd.DataFrame(nmf_topic_rows)
nmf_topic_summary.to_csv(
    os.path.join(EVAL_DIR, "definitional_nmf_topic_candidates.csv"),
    index=False,
)
print("Six-topic NMF candidate solution:")
display(nmf_topic_summary)


,n_topics,mean_aligned_topic_similarity,topic_diversity,smallest_topic_share,reconstruction_error
0,3,1.0,0.911,0.111,137.251
1,4,1.0,0.933,0.086,136.369
2,5,1.0,0.893,0.083,135.691
3,6,1.0,0.911,0.078,135.046
4,7,1.0,0.867,0.044,134.542
5,8,1.0,0.833,0.035,134.054


alt.Chart(...)

Six-topic NMF candidate solution:


,topic,n_spans,pct_spans,top_terms,representative_excerpt,largest_service_share,largest_service,largest_prompt_share,largest_prompt
0,NMF 1,2173,10.517400,culture; human culture; study human; human; st...,"'Humanities' is translated as 'ɓeɓeɓeɗ', meani...",0.215370,gemini,0.508053,minimal
1,NMF 2,1606,7.773099,translates; translates translates; directly; d...,"In Lojban, 'digitali' translates to 'digital',...",0.528020,mistral,0.455791,github_searcher
2,NMF 3,2374,11.490247,combines; interdisciplinary field; field combi...,The term 'Digital Humanities' refers to the in...,0.290227,openai,0.477254,judge
3,NMF 4,6206,30.037268,academic; direct; used; academic field; common...,This translation combines 'Egesaku kia' (the f...,0.183532,deepseek,0.435063,github_searcher
4,NMF 5,1627,7.874740,refers; intersection; intersection technology;...,"The term 'tinklinys' means digital, and 'human...",0.232329,qwen,0.362016,judge
5,NMF 6,6675,32.307246,knowledge; learning; knowledge learning; human...,'Dhut' means 'knowledge' or 'learning'.,0.184869,gemma,0.402247,judge


### BERTopic-Style Embedding Clusters

The embedding analysis asks whether the spans form discrete semantic neighborhoods without fixing the number of topics in advance. I encode the cleaned spans with `all-MiniLM-L6-v2`, reduce the embeddings to five dimensions with UMAP, and cluster them with HDBSCAN. I repeat HDBSCAN at three minimum-cluster sizes. Large changes in cluster count, outlier share, or adjusted Rand agreement indicate that the apparent boundaries are parameter-sensitive. This is especially important here because density clustering can turn templates, scripts, or repeated model language into seemingly substantive topics.


In [5]:
import hashlib
import tempfile

os.environ.setdefault("NUMBA_CACHE_DIR", str(Path(tempfile.gettempdir()) / "coding-dh-numba-cache"))
from hdbscan import HDBSCAN
from sentence_transformers import SentenceTransformer
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
import umap

span_key = "␟".join(definitional_spans["span_clean"].tolist())
span_digest = hashlib.sha256(span_key.encode("utf-8")).hexdigest()[:16]
span_embedding_cache = Path("eda_cache") / f"definitional_span_embeddings_{span_digest}.npy"
span_embedding_cache.parent.mkdir(exist_ok=True)

if span_embedding_cache.exists():
    definitional_embeddings = np.load(span_embedding_cache)
else:
    definition_model = SentenceTransformer("all-MiniLM-L6-v2", local_files_only=True)
    definitional_embeddings = definition_model.encode(
        definitional_spans["span_clean"].tolist(),
        batch_size=128,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    np.save(span_embedding_cache, definitional_embeddings)

definition_umap = umap.UMAP(
    n_neighbors=30,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42,
)
definition_reduced = definition_umap.fit_transform(definitional_embeddings)

embedding_cluster_labels = {}
embedding_sensitivity_rows = []
for min_cluster_size in [100, 200, 350]:
    labels = HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=20,
        metric="euclidean",
        cluster_selection_method="eom",
    ).fit_predict(definition_reduced)
    embedding_cluster_labels[min_cluster_size] = labels
    embedding_sensitivity_rows.append({
        "min_cluster_size": min_cluster_size,
        "clusters": len(set(labels) - {-1}),
        "outlier_pct": float((labels == -1).mean() * 100),
    })

embedding_sensitivity = pd.DataFrame(embedding_sensitivity_rows)
embedding_ari_rows = []
settings = sorted(embedding_cluster_labels)
for left_index, left in enumerate(settings):
    for right in settings[left_index + 1:]:
        embedding_ari_rows.append({
            "comparison": f"{left} vs {right}",
            "adjusted_rand": adjusted_rand_score(
                embedding_cluster_labels[left],
                embedding_cluster_labels[right],
            ),
        })
embedding_ari = pd.DataFrame(embedding_ari_rows)

display(embedding_sensitivity.round(3))
display(embedding_ari.round(3))

cluster_count_chart = alt.Chart(embedding_sensitivity).mark_bar(color="#457B9D").encode(
    x=alt.X("min_cluster_size:O", title="Minimum cluster size"),
    y=alt.Y("clusters:Q", title="Clusters found"),
    tooltip=["min_cluster_size:O", "clusters:Q"],
).properties(width=245, height=235, title="Cluster count")
outlier_chart = alt.Chart(embedding_sensitivity).mark_bar(color="#E76F51").encode(
    x=alt.X("min_cluster_size:O", title="Minimum cluster size"),
    y=alt.Y("outlier_pct:Q", title="Spans labeled as outliers (%)"),
    tooltip=["min_cluster_size:O", alt.Tooltip("outlier_pct:Q", format=".1f")],
).properties(width=245, height=235, title="Outlier rate")

embedding_sensitivity_chart = alt.hconcat(cluster_count_chart, outlier_chart).properties(
    title=alt.TitleParams(
        "BERTopic-Style Density Clusters Are Parameter-Sensitive",
        subtitle="Sentence embeddings + five-dimensional UMAP + HDBSCAN.",
        anchor="start",
    )
).configure_view(stroke=None)

save_chart(
    embedding_sensitivity_chart,
    str(FIGURE_DIR / "definitional_embedding_cluster_sensitivity.png"),
    scale_factor=2.0,
)
display(embedding_sensitivity_chart)

# Summarize the middle setting with class-level TF-IDF terms and concentration diagnostics.
EMBEDDING_CANDIDATE_SETTING = 200
candidate_cluster_labels = embedding_cluster_labels[EMBEDDING_CANDIDATE_SETTING]
definitional_spans["embedding_cluster"] = candidate_cluster_labels
cluster_ids = sorted(set(candidate_cluster_labels) - {-1})
cluster_documents = [
    " ".join(definitional_spans.loc[candidate_cluster_labels == cluster_id, "span_clean"])
    for cluster_id in cluster_ids
]
cluster_vectorizer = TfidfVectorizer(
    stop_words=topic_stopwords,
    ngram_range=(1, 2),
    max_features=20_000,
)
cluster_tfidf = cluster_vectorizer.fit_transform(cluster_documents)
cluster_terms = np.asarray(cluster_vectorizer.get_feature_names_out())

embedding_topic_rows = []
for position, cluster_id in enumerate(cluster_ids):
    indices = np.where(candidate_cluster_labels == cluster_id)[0]
    top_ids = cluster_tfidf[position].toarray().ravel().argsort()[-10:][::-1]
    service_shares = definitional_spans.iloc[indices]["service"].value_counts(normalize=True)
    prompt_shares = definitional_spans.iloc[indices]["variant"].value_counts(normalize=True)
    embedding_topic_rows.append({
        "cluster": cluster_id,
        "n_spans": len(indices),
        "top_terms": "; ".join(cluster_terms[top_ids]),
        "largest_service": service_shares.index[0],
        "largest_service_share": float(service_shares.iloc[0]),
        "largest_prompt": prompt_shares.index[0],
        "largest_prompt_share": float(prompt_shares.iloc[0]),
        "representative_excerpt": definitional_spans.iloc[indices[0]]["span_raw"],
    })

embedding_topic_summary = pd.DataFrame(embedding_topic_rows)
embedding_topic_summary.to_csv(
    os.path.join(EVAL_DIR, "definitional_embedding_topic_candidates.csv"),
    index=False,
)

non_outlier = candidate_cluster_labels != -1
nmf_embedding_nmi = normalized_mutual_info_score(
    candidate_assignments[non_outlier],
    candidate_cluster_labels[non_outlier],
)
service_concentrated = int((embedding_topic_summary["largest_service_share"] >= 0.50).sum())
prompt_concentrated = int((embedding_topic_summary["largest_prompt_share"] >= 0.70).sum())

print(f"NMF-topic vs embedding-cluster NMI (non-outliers): {nmf_embedding_nmi:.3f}")
print(
    f"Embedding clusters with >=50% of spans from one service: "
    f"{service_concentrated}/{len(embedding_topic_summary)}"
)
print(
    f"Embedding clusters with >=70% of spans from one prompt: "
    f"{prompt_concentrated}/{len(embedding_topic_summary)}"
)
display(
    embedding_topic_summary.sort_values("n_spans", ascending=False).head(12)
)


<environment>/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


,min_cluster_size,clusters,outlier_pct
0,100,42,29.079
1,200,22,31.973
2,350,3,5.000


,comparison,adjusted_rand
0,100 vs 200,0.776
1,100 vs 350,0.036
2,200 vs 350,0.035


alt.HConcatChart(...)

NMF-topic vs embedding-cluster NMI (non-outliers): 0.158
Embedding clusters with >=50% of spans from one service: 2/22
Embedding clusters with >=70% of spans from one prompt: 3/22


,cluster,n_spans,top_terms,largest_service,largest_service_share,largest_prompt,largest_prompt_share,representative_excerpt
9,9,2838,human; study; knowledge; culture; refers; huma...,gemini,0.285060,minimal,0.382311,This captures the concept of 'humanities' as t...
11,11,1544,knowledge; field; study; methods; tools; inter...,gemma,0.303756,judge,0.417746,The term combines the concept of humanities/hu...
2,2,1542,academic; specialized; direct; like; specializ...,openai,0.289883,github_searcher,0.453956,The term 'Digital Humanities' is a modern acad...
21,21,794,knowledge; human; technology; combines; study;...,deepseek,0.293451,github_searcher,0.409320,"However, a more accurate modern translation wo..."
13,13,779,refers; knowledge; human; electronic; study; t...,qwen,0.320924,minimal,0.415918,"In Bemba, 'ubuntungwa' refers to humanities/hu..."
14,14,736,uses; accurately; combines; combining; concept...,qwen,0.245924,judge,0.695652,An alternative shorter form could be 'Tussniwi...
20,20,611,combining; combines; standard combining; stand...,qwen,0.237316,github_searcher,0.463175,This translation breaks down as follows: 'Ilim...
5,5,536,technology; used; transliteration; direct; dir...,gemma,0.257463,minimal,0.423507,'ਡਿਜੀਟਲ' (Digital) is a direct transliteration...
8,8,527,hindi; navik; nepali; devanagari script; devan...,deepseek,0.277040,github_searcher,0.402277,This translation combines 'āṅgika' (आङ्गिक) me...
18,18,479,combining; words; translates; combining words;...,llama,0.382046,github_searcher,0.557411,"In Dimli (Zazaki), 'Digital' is adapted as 'Dî..."


### Reading the Inductive Results

The two methods answer different questions. NMF identifies recurring vocabularies that can overlap within a rationale and is therefore the better basis for proposing transparent lexical rules. The embedding-density pipeline identifies locally repeated formulations, but its cluster count and outlier rate are highly sensitive to the density threshold. Clusters concentrated in one service or prompt are better interpreted as templates or rhetorical genres than as definitions of DH.

I therefore use the six-topic NMF solution as a deliberately granular candidate inventory rather than accepting either model's assignments as final categories. The later rule-based section remains the transparent operationalization, while the NMF terms and excerpts provide evidence for deciding whether those rules should be split, merged, or reclassified as translation strategies rather than definitions.


## 7.3 — Topic-Informed Frames and Authorization Strategies

The six-topic NMF solution separates two kinds of language that the original four-register scheme partly mixed together. Four topics describe the disciplinary object: DH as an interdisciplinary field, as an intersection of technical methods and humanities inquiry, as the study of human culture and society, or as knowledge and learning. Two additional topics describe how the model authorizes a translation: through lexical composition or through claims about established academic usage. I translate those six recurring vocabularies into transparent, overlapping rules below.

These are **topic-informed signals**, not automatic topic labels. NMF supplied terms and representative excerpts; I converted those patterns into rules that can be inspected, rerun, and compared across services and prompts. The distinction between definitional frames and authorization strategies prevents a literal word-by-word explanation from being treated as the same kind of claim as a definition of the discipline.


In [6]:
TOPIC_INFORMED_PATTERNS = {
    "interdisciplinary_field": [
        r"\binterdisciplinary\s+(?:field|study|research|approach)\b",
        r"\b(?:academic|scholarly)\s+(?:field|discipline)\b",
        r"\bfield\s+that\s+combines?\b",
    ],
    "technical_methodological_intersection": [
        r"\bintersection\s+of\s+(?:digital|technology|comput)",
        r"\b(?:digital|computational)\s+(?:methods?|tools?)\b",
        r"\b(?:technology|digital tools?)\s+(?:to|for|in)\s+(?:study|research|analy)",
    ],
    "human_cultural_scope": [
        r"\bhuman\s+(?:culture|experience|society|history)\b",
        r"\bstudy\s+of\s+human\s+culture\b",
        r"\bculture\s+and\s+society\b",
    ],
    "knowledge_learning_scope": [
        r"\b(?:humanities|humanistic|human)\s+(?:knowledge|learning|wisdom)\b",
        r"\bknowledge\s+(?:or|and)\s+learning\b",
        r"\bdigital\b.*\b(?:knowledge|education|learning|wisdom)\b",
        r"\b(?:knowledge|education|learning|wisdom)\b.*\bdigital\b",
    ],
    "lexical_compositional_strategy": [
        r"\bcombines?\b.{0,100}\bdigital\b",
        r"\bdigital\b.{0,100}\b(?:and|with|plus)\b.{0,100}\bhumanities\b",
        r"\b(?:compound|word.for.word|literal(?:ly)?)\b",
    ],
    "institutional_usage_legitimacy": [
        r"\b(?:standard|established|accepted|recognized|recognised)\s+(?:term|translation|equivalent)\b",
        r"\b(?:widely|commonly)\s+(?:used|understood|accepted|recognized|recognised)\b",
        r"\b(?:academic|scholarly)\s+(?:usage|contexts?|communities|literature)\b",
    ],
}

TOPIC_FRAME_LABELS = {
    "interdisciplinary_field": "Interdisciplinary field",
    "technical_methodological_intersection": "Technical-methodological intersection",
    "human_cultural_scope": "Human-cultural scope",
    "knowledge_learning_scope": "Knowledge-learning scope",
    "lexical_compositional_strategy": "Lexical-compositional strategy",
    "institutional_usage_legitimacy": "Institutional usage/legitimacy",
}
TOPIC_FRAME_LAYERS = {
    "interdisciplinary_field": "Definition of DH",
    "technical_methodological_intersection": "Definition of DH",
    "human_cultural_scope": "Definition of DH",
    "knowledge_learning_scope": "Definition of DH",
    "lexical_compositional_strategy": "Translation authorization",
    "institutional_usage_legitimacy": "Translation authorization",
}

def matches_topic_frame(rationale, frame):
    return any(
        re.search(pattern, rationale, re.IGNORECASE)
        for pattern in TOPIC_INFORMED_PATTERNS[frame]
    )

for frame in TOPIC_INFORMED_PATTERNS:
    rationale_df[f"topic_frame_{frame}"] = rationale_df["rationale"].apply(
        lambda rationale, frame=frame: matches_topic_frame(rationale, frame)
    )
    definitional_spans[f"topic_frame_{frame}"] = rationale_df[f"topic_frame_{frame}"].to_numpy()

topic_frame_order = list(TOPIC_INFORMED_PATTERNS)
topic_frame_display_order = [TOPIC_FRAME_LABELS[frame] for frame in topic_frame_order]

topic_frame_service = (
    rationale_df.groupby("service")[[f"topic_frame_{frame}" for frame in topic_frame_order]]
    .mean()
    .mul(100)
    .reset_index()
    .melt(id_vars="service", var_name="frame", value_name="pct")
)
topic_frame_service["frame"] = topic_frame_service["frame"].str.replace("topic_frame_", "", regex=False)
topic_frame_service["frame_label"] = topic_frame_service["frame"].map(TOPIC_FRAME_LABELS)
topic_frame_service["layer"] = topic_frame_service["frame"].map(TOPIC_FRAME_LAYERS)
topic_frame_service["model"] = topic_frame_service["service"].map({
    "claude": "Claude", "openai": "OpenAI", "gemini": "Gemini", "deepseek": "DeepSeek",
    "llama": "Llama", "gemma": "Gemma", "qwen": "Qwen", "mistral": "Mistral",
})
topic_model_order = ["Claude", "OpenAI", "Gemini", "DeepSeek", "Llama", "Gemma", "Qwen", "Mistral"]

topic_frame_base = alt.Chart(topic_frame_service).encode(
    x=alt.X(
        "frame_label:N",
        sort=topic_frame_display_order,
        title=None,
        axis=alt.Axis(labelAngle=-35),
    ),
    y=alt.Y("model:N", sort=topic_model_order, title=None),
    tooltip=[
        alt.Tooltip("model:N", title="Model"),
        alt.Tooltip("frame_label:N", title="Signal"),
        alt.Tooltip("layer:N", title="Analytical layer"),
        alt.Tooltip("pct:Q", title="% of rationales", format=".1f"),
    ],
)
topic_frame_rect = topic_frame_base.mark_rect(stroke="white", strokeWidth=2).encode(
    color=alt.Color("pct:Q", scale=alt.Scale(scheme="viridis", domain=[0, 100]), title="% of rationales")
)
topic_frame_text = topic_frame_base.mark_text(fontSize=9).encode(
    text=alt.Text("pct:Q", format=".0f"),
    color=alt.condition(alt.datum.pct > 55, alt.value("white"), alt.value("black")),
)
topic_frame_service_chart = (topic_frame_rect + topic_frame_text).properties(
    width=620,
    height=300,
    title=alt.TitleParams(
        "Topic-Informed Definitional and Authorization Signals by Service",
        subtitle="Signals overlap; each cell is the percentage of a service's rationales matching the rule.",
        anchor="start",
    ),
).configure_view(stroke=None)
save_chart(
    topic_frame_service_chart,
    str(FIGURE_DIR / "topic_informed_frames_by_service.png"),
    scale_factor=2.0,
)
display(topic_frame_service_chart)

# Crosswalk the six NMF assignments with the six transparent signals.
crosswalk_rows = []
for topic_id in range(NMF_CANDIDATE_K):
    topic_mask = definitional_spans["nmf_topic"].eq(topic_id)
    for frame in topic_frame_order:
        crosswalk_rows.append({
            "nmf_topic": f"NMF {topic_id + 1}",
            "frame": TOPIC_FRAME_LABELS[frame],
            "pct": definitional_spans.loc[topic_mask, f"topic_frame_{frame}"].mean() * 100,
            "n_topic": int(topic_mask.sum()),
        })
topic_frame_crosswalk = pd.DataFrame(crosswalk_rows)

crosswalk_base = alt.Chart(topic_frame_crosswalk).encode(
    x=alt.X("frame:N", sort=topic_frame_display_order, title=None, axis=alt.Axis(labelAngle=-35)),
    y=alt.Y("nmf_topic:N", sort=[f"NMF {i}" for i in range(1, 7)], title=None),
    tooltip=["nmf_topic:N", "frame:N", "n_topic:Q", alt.Tooltip("pct:Q", format=".1f")],
)
crosswalk_rect = crosswalk_base.mark_rect(stroke="white", strokeWidth=2).encode(
    color=alt.Color("pct:Q", scale=alt.Scale(scheme="blues", domain=[0, 100]), title="% of NMF topic")
)
crosswalk_text = crosswalk_base.mark_text(fontSize=9).encode(
    text=alt.Text("pct:Q", format=".0f"),
    color=alt.condition(alt.datum.pct > 55, alt.value("white"), alt.value("black")),
)
topic_frame_crosswalk_chart = (crosswalk_rect + crosswalk_text).properties(
    width=620,
    height=250,
    title=alt.TitleParams(
        "NMF Topics and Topic-Informed Rule Signals",
        subtitle="Cells show the share of spans assigned to each NMF topic that match each transparent rule.",
        anchor="start",
    ),
).configure_view(stroke=None)
save_chart(
    topic_frame_crosswalk_chart,
    str(FIGURE_DIR / "nmf_topic_rule_crosswalk.png"),
    scale_factor=2.0,
)
display(topic_frame_crosswalk_chart)


alt.LayerChart(...)

alt.LayerChart(...)

### Rationale-Level Distribution and Overlap

Because the six categories are overlapping signals, I do not assign each rationale to one dominant class. I instead measure how many signals each rationale activates, which exact combinations recur, and how often each pair co-occurs. The first chart shows the distribution from zero through six matched signals. The second reports pairwise overlap as the percentage of all retained rationales matching both signals. A zero-match rationale is not necessarily definition-free; it may use phrasing outside the transparent rule lists.


In [7]:
topic_signal_cols = [f"topic_frame_{frame}" for frame in topic_frame_order]
rationale_df["topic_signal_count"] = rationale_df[topic_signal_cols].sum(axis=1).astype(int)

def _signal_combination(row):
    matched = [
        TOPIC_FRAME_LABELS[frame]
        for frame in topic_frame_order
        if bool(row[f"topic_frame_{frame}"])
    ]
    return " + ".join(matched) if matched else "No rule match"

rationale_df["topic_signal_combination"] = rationale_df.apply(_signal_combination, axis=1)

signal_count_distribution = (
    rationale_df["topic_signal_count"]
    .value_counts()
    .reindex(range(len(topic_frame_order) + 1), fill_value=0)
    .rename_axis("signals_matched")
    .reset_index(name="rationales")
)
signal_count_distribution["pct"] = (
    signal_count_distribution["rationales"] / len(rationale_df) * 100
)
signal_count_distribution["label"] = signal_count_distribution.apply(
    lambda row: f"{int(row['rationales']):,} ({row['pct']:.1f}%)", axis=1
)

count_bars = alt.Chart(signal_count_distribution).mark_bar(color="#2A9D8F").encode(
    x=alt.X("signals_matched:O", title="Number of overlapping categories matched"),
    y=alt.Y("rationales:Q", title="Rationales"),
    tooltip=["signals_matched:O", "rationales:Q", alt.Tooltip("pct:Q", format=".1f")],
)
count_labels = count_bars.mark_text(dy=-8, color="#111111", fontSize=10).encode(
    text="label:N"
)
topic_signal_count_chart = (count_bars + count_labels).properties(
    width=570,
    height=300,
    title=alt.TitleParams(
        "How Many Topic-Informed Categories Does Each Rationale Match?",
        subtitle="Categories overlap; zero matches can include paraphrases outside the transparent rules.",
        anchor="start",
    ),
).configure_view(stroke=None)
save_chart(
    topic_signal_count_chart,
    str(FIGURE_DIR / "topic_informed_categories_per_rationale.png"),
    scale_factor=2.0,
)
display(topic_signal_count_chart)

top_signal_combinations = (
    rationale_df["topic_signal_combination"]
    .value_counts()
    .rename_axis("category_combination")
    .reset_index(name="rationales")
)
top_signal_combinations["pct"] = top_signal_combinations["rationales"] / len(rationale_df) * 100
print("Most common category combinations:")
display(top_signal_combinations.head(15).round({"pct": 1}))

pair_rows = []
for row_frame in topic_frame_order:
    for col_frame in topic_frame_order:
        both = (
            rationale_df[f"topic_frame_{row_frame}"]
            & rationale_df[f"topic_frame_{col_frame}"]
        ).sum()
        pair_rows.append({
            "row_signal": TOPIC_FRAME_LABELS[row_frame],
            "column_signal": TOPIC_FRAME_LABELS[col_frame],
            "rationales": int(both),
            "pct_all": both / len(rationale_df) * 100,
        })
topic_signal_pairwise = pd.DataFrame(pair_rows)

pair_base = alt.Chart(topic_signal_pairwise).encode(
    x=alt.X(
        "column_signal:N", sort=topic_frame_display_order, title=None,
        axis=alt.Axis(labelAngle=-35),
    ),
    y=alt.Y("row_signal:N", sort=topic_frame_display_order, title=None),
    tooltip=[
        alt.Tooltip("row_signal:N", title="Signal 1"),
        alt.Tooltip("column_signal:N", title="Signal 2"),
        alt.Tooltip("rationales:Q", title="Rationales"),
        alt.Tooltip("pct_all:Q", title="% of all rationales", format=".1f"),
    ],
)
pair_rect = pair_base.mark_rect(stroke="white", strokeWidth=2).encode(
    color=alt.Color(
        "pct_all:Q", scale=alt.Scale(scheme="tealblues"), title="% of all rationales"
    )
)
pair_text = pair_base.mark_text(fontSize=9).encode(
    text=alt.Text("pct_all:Q", format=".1f"),
    color=alt.condition(alt.datum.pct_all > 30, alt.value("white"), alt.value("black")),
)
topic_signal_overlap_chart = (pair_rect + pair_text).properties(
    width=560,
    height=360,
    title=alt.TitleParams(
        "Pairwise Overlap Among Topic-Informed Categories",
        subtitle="Each cell is the percentage of all retained rationales matching both signals; diagonal cells show prevalence.",
        anchor="start",
    ),
).configure_view(stroke=None)
save_chart(
    topic_signal_overlap_chart,
    str(FIGURE_DIR / "topic_informed_category_overlap.png"),
    scale_factor=2.0,
)
display(topic_signal_overlap_chart)


alt.LayerChart(...)

Most common category combinations:


,category_combination,rationales,pct
0,Lexical-compositional strategy,4581,22.2
1,No rule match,4158,20.1
2,Lexical-compositional strategy + Institutional...,1828,8.8
3,Institutional usage/legitimacy,1645,8.0
4,Knowledge-learning scope + Lexical-composition...,1063,5.1
5,Knowledge-learning scope,779,3.8
6,Technical-methodological intersection + Lexica...,611,3.0
7,Interdisciplinary field + Lexical-compositiona...,536,2.6
8,Interdisciplinary field + Lexical-compositiona...,428,2.1
9,Human-cultural scope + Lexical-compositional s...,384,1.9


alt.LayerChart(...)

In [8]:
from great_tables import GT

topic_schema_examples = [
    {
        "frame": "Interdisciplinary field",
        "layer": "Definition of DH",
        "detects": "Presents DH as an interdisciplinary academic field or discipline.",
        "service": "deepseek", "variant": "github_searcher", "language_code": "xal",
        "bold": ["interdisciplinary field"],
    },
    {
        "frame": "Technical-methodological intersection",
        "layer": "Definition of DH",
        "detects": "Defines DH through digital methods or tools applied to humanities inquiry.",
        "service": "llama", "variant": "minimal", "language_code": "rob",
        "bold": ["digital methods", "humanities research"],
    },
    {
        "frame": "Human-cultural scope",
        "layer": "Definition of DH",
        "detects": "Defines the humanities through people, culture, society, experience, or history.",
        "service": "mistral", "variant": "github_searcher", "language_code": "mwr",
        "bold": ["study of human culture"],
    },
    {
        "frame": "Knowledge-learning scope",
        "layer": "Definition of DH",
        "detects": "Defines the humanities as knowledge, learning, education, or wisdom.",
        "service": "mistral", "variant": "minimal", "language_code": "lmn",
        "bold": ["knowledge"],
    },
    {
        "frame": "Lexical-compositional strategy",
        "layer": "Translation authorization",
        "detects": "Justifies a term through literal translation, compounding, or component combination.",
        "service": "mistral", "variant": "minimal", "language_code": "tab",
        "bold": ["Digital", "Humanities"],
    },
    {
        "frame": "Institutional usage/legitimacy",
        "layer": "Translation authorization",
        "detects": "Authorizes a term by claiming standard, established, recognized, or academic usage.",
        "service": "qwen", "variant": "github_searcher", "language_code": "ur",
        "bold": ["standard translation", "academic and digital contexts"],
    },
]

def bold_schema_phrases(text, phrases):
    marked = text
    for phrase in phrases:
        marked = re.sub(
            re.escape(phrase),
            lambda match: f"**{match.group(0)}**",
            marked,
            count=1,
            flags=re.IGNORECASE,
        )
    return marked

schema_table_rows = []
for example in topic_schema_examples:
    match = rationale_df[
        rationale_df["service"].eq(example["service"])
        & rationale_df["variant"].eq(example["variant"])
        & rationale_df["language_code"].eq(example["language_code"])
    ]
    if len(match) != 1:
        raise ValueError(f"Expected one rationale for schema example; found {len(match)}")
    source_row = explorer_lookup.loc[example["language_code"]]
    if isinstance(source_row, pd.DataFrame):
        source_row = source_row.iloc[0]
    schema_table_rows.append({
        "layer": example["layer"],
        "frame": example["frame"],
        "detects": example["detects"],
        "example": bold_schema_phrases(match.iloc[0]["rationale"], example["bold"]),
        "source": (
            f"{example['service'].title() if example['service'] != 'deepseek' else 'DeepSeek'}; "
            f"{'GitHub searcher' if example['variant'] == 'github_searcher' else example['variant'].title()}; "
            f"{source_row.get('language_name', '')} ({example['language_code']})"
        ),
    })

topic_schema_table_df = pd.DataFrame(schema_table_rows)
topic_schema_table = (
    GT(topic_schema_table_df)
    .tab_header(
        title="Topic-Informed Signals for Reading LLM Rationales",
        subtitle="Four signals describe DH; two describe how models authorize a proposed translation.",
    )
    .cols_label(
        layer="Analytical layer",
        frame="Signal",
        detects="What the rule detects",
        example="Actual rationale excerpt",
        source="Source",
    )
    .fmt_markdown(columns="example")
    .cols_width(
        layer="175px", frame="220px", detects="300px", example="500px", source="180px"
    )
    .tab_options(
        column_labels_font_weight="bold",
        column_labels_vlines_style="solid",
        column_labels_vlines_width="1px",
        column_labels_vlines_color="#A9A9A9",
        table_body_vlines_style="solid",
        table_body_vlines_width="1px",
        table_body_vlines_color="#D0D0D0",
        table_body_hlines_style="solid",
        table_body_hlines_width="1px",
        table_body_hlines_color="#D9D9D9",
    )
    .tab_source_note(
        source_note=(
            "Signals were derived from stable NMF vocabularies and translated into inspectable lexical rules. "
            "Examples illustrate rule activation, not linguistic validity."
        )
    )
)
display(topic_schema_table)
topic_schema_table.gtsave(str(FIGURE_DIR / "topic_informed_schema_table.png"))


GT(_tbl_data=                       layer                                  frame  \
0           Definition of DH                Interdisciplinary field   
1           Definition of DH  Technical-methodological intersection   
2           Definition of DH                   Human-cultural scope   
3           Definition of DH               Knowledge-learning scope   
4  Translation authorization         Lexical-compositional strategy   
5  Translation authorization         Institutional usage/legitimacy   

                                             detects  \
0  Presents DH as an interdisciplinary academic f...   
1  Defines DH through digital methods or tools ap...   
2  Defines the humanities through people, culture...   
3  Defines the humanities as knowledge, learning,...   
4  Justifies a term through literal translation, ...   
5  Authorizes a term by claiming standard, establ...   

                                             example  \
0  This translation combines 'Цифрин' (digital) w...   
1  A field that combines **digital methods** with...   
2  Digital Humanities in English translates to 'O...   
3  Using approximations of 'digital' (computer) a...   
4  Translation of individual words '**Digital**' ...   
5  This is the **standard translation** of 'Digit...   

                                    source  
0  DeepSeek; GitHub searcher; Kalmyk (xal)  
1               Llama; Minimal; Tae' (rob)  
2  Mistral; GitHub searcher; Marwari (mwr)  
3          Mistral; Minimal; Lambadi (lmn)  
4       Mistral; Minimal; Tabassaran (tab)  
5         Qwen; GitHub searcher; Urdu (ur)  , _body=<great_tables._gt_data.Body object at 0x14aa6a660>, _boxhead=Boxhead([ColInfo(var='layer', type=<ColInfoTypeEnum.default: 1>, column_label='Analytical layer', column_align='left', column_width='175px'), ColInfo(var='frame', type=<ColInfoTypeEnum.default: 1>, column_label='Signal', column_align='left', column_width='220px'), ColInfo(var='detects', type=<ColInfoTypeEnum.default: 1>, column_label='What the rule detects', column_align='left', column_width='300px'), ColInfo(var='example', type=<ColInfoTypeEnum.default: 1>, column_label='Actual rationale excerpt', column_align='left', column_width='500px'), ColInfo(var='source', type=<ColInfoTypeEnum.default: 1>, column_label='Source', column_align='left', column_width='180px')]), _stub=<great_tables._gt_data.Stub object at 0x14aa6a7b0>, _spanners=Spanners([]), _heading=Heading(title='Topic-Informed Signals for Reading LLM Rationales', subtitle='Four signals describe DH; two describe how models authorize a proposed translation.', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x14aa6aba0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x14aab96d0>, _source_notes=['Signals were derived from stable NMF vocabularies and translated into inspectable lexical rules. Examples illustrate rule activation, not linguistic validity.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x14aa6acf0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x14aa6aa50>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Seg

GT(_tbl_data=                       layer                                  frame  \
0           Definition of DH                Interdisciplinary field   
1           Definition of DH  Technical-methodological intersection   
2           Definition of DH                   Human-cultural scope   
3           Definition of DH               Knowledge-learning scope   
4  Translation authorization         Lexical-compositional strategy   
5  Translation authorization         Institutional usage/legitimacy   

                                             detects  \
0  Presents DH as an interdisciplinary academic f...   
1  Defines DH through digital methods or tools ap...   
2  Defines the humanities through people, culture...   
3  Defines the humanities as knowledge, learning,...   
4  Justifies a term through literal translation, ...   
5  Authorizes a term by claiming standard, establ...   

                                             example  \
0  This translation combines 'Цифрин' (digital) w...   
1  A field that combines **digital methods** with...   
2  Digital Humanities in English translates to 'O...   
3  Using approximations of 'digital' (computer) a...   
4  Translation of individual words '**Digital**' ...   
5  This is the **standard translation** of 'Digit...   

                                    source  
0  DeepSeek; GitHub searcher; Kalmyk (xal)  
1               Llama; Minimal; Tae' (rob)  
2  Mistral; GitHub searcher; Marwari (mwr)  
3          Mistral; Minimal; Lambadi (lmn)  
4       Mistral; Minimal; Tabassaran (tab)  
5         Qwen; GitHub searcher; Urdu (ur)  , _body=<great_tables._gt_data.Body object at 0x14aa6a660>, _boxhead=Boxhead([ColInfo(var='layer', type=<ColInfoTypeEnum.default: 1>, column_label='Analytical layer', column_align='left', column_width='175px'), ColInfo(var='frame', type=<ColInfoTypeEnum.default: 1>, column_label='Signal', column_align='left', column_width='220px'), ColInfo(var='detects', type=<ColInfoTypeEnum.default: 1>, column_label='What the rule detects', column_align='left', column_width='300px'), ColInfo(var='example', type=<ColInfoTypeEnum.default: 1>, column_label='Actual rationale excerpt', column_align='left', column_width='500px'), ColInfo(var='source', type=<ColInfoTypeEnum.default: 1>, column_label='Source', column_align='left', column_width='180px')]), _stub=<great_tables._gt_data.Stub object at 0x14aa6a7b0>, _spanners=Spanners([]), _heading=Heading(title='Topic-Informed Signals for Reading LLM Rationales', subtitle='Four signals describe DH; two describe how models authorize a proposed translation.', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x14aa6aba0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x14aab96d0>, _source_notes=['Signals were derived from stable NMF vocabularies and translated into inspectable lexical rules. Examples illustrate rule activation, not linguistic validity.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x14aa6acf0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x14aa6aa50>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Seg

### Robustness of the Topic-Informed Rules

I repeat the grouped multilabel classification test for the six topic-informed signals. The unmasked model measures how consistently the rules map onto the surrounding vocabulary. The masked model deletes the principal terms used to construct all six rules. Persistence after masking indicates broader rhetorical coherence, while a large decline indicates that a signal is primarily lexical. As before, this remains a test against rule-derived labels rather than human semantic judgments.


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_fscore_support
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline

TOPIC_RULE_TRIGGER_MASK = re.compile(
    r"\b(?:interdisciplinary|academic|scholarly|field|discipline|digital|computational|methods?|"
    r"tools?|technology|human|culture|experience|society|history|knowledge|learning|education|wisdom|"
    r"combines?|compound|literal(?:ly)?|standard|established|accepted|recognized|recognised|widely|"
    r"commonly|used|understood|usage|contexts?|communities|literature|humanities)\b",
    re.IGNORECASE,
)

topic_rule_targets = rationale_df[
    [f"topic_frame_{frame}" for frame in topic_frame_order]
].astype(int).to_numpy()
topic_rule_groups = rationale_df["language_code"].to_numpy()
topic_rule_conditions = {
    "Original rationales": rationale_df["rationale"].tolist(),
    "Trigger-masked rationales": rationale_df["rationale"].map(
        lambda text: re.sub(r"\s+", " ", TOPIC_RULE_TRIGGER_MASK.sub(" ", text)).strip()
    ).tolist(),
}

topic_rule_metric_rows = []
for condition, texts in topic_rule_conditions.items():
    estimator = Pipeline([
        (
            "tfidf",
            TfidfVectorizer(
                ngram_range=(1, 2), min_df=5, max_features=50_000, sublinear_tf=True
            ),
        ),
        (
            "classifier",
            OneVsRestClassifier(
                LogisticRegression(
                    max_iter=2_000, class_weight="balanced", solver="liblinear"
                )
            ),
        ),
    ])
    predictions = cross_val_predict(
        estimator,
        texts,
        topic_rule_targets,
        groups=topic_rule_groups,
        cv=GroupKFold(5),
        method="predict",
        n_jobs=-1,
    )
    precision, recall, f1, support = precision_recall_fscore_support(
        topic_rule_targets, predictions, average=None, zero_division=0
    )
    for frame, p, r, f, n in zip(topic_frame_order, precision, recall, f1, support):
        topic_rule_metric_rows.append({
            "condition": condition,
            "frame": TOPIC_FRAME_LABELS[frame],
            "layer": TOPIC_FRAME_LAYERS[frame],
            "precision": p,
            "recall": r,
            "f1": f,
            "positive_rationales": int(n),
        })

topic_rule_metrics = pd.DataFrame(topic_rule_metric_rows)
topic_rule_macro = topic_rule_metrics.groupby("condition")[["precision", "recall", "f1"]].mean()
print("Topic-informed rule classifier performance:")
display(topic_rule_macro.round(3))
display(topic_rule_metrics.round(3))

topic_rule_condition_order = ["Original rationales", "Trigger-masked rationales"]
topic_rule_bars = alt.Chart(topic_rule_metrics).mark_bar().encode(
    y=alt.Y("frame:N", sort=topic_frame_display_order, title=None),
    yOffset=alt.YOffset("condition:N", sort=topic_rule_condition_order),
    x=alt.X("f1:Q", title="Out-of-fold F1", scale=alt.Scale(domain=[0, 1])),
    color=alt.Color(
        "condition:N",
        sort=topic_rule_condition_order,
        title=None,
        scale=alt.Scale(domain=topic_rule_condition_order, range=["#2A9D8F", "#E76F51"]),
    ),
    tooltip=[
        "layer:N", "frame:N", "condition:N",
        alt.Tooltip("precision:Q", format=".3f"),
        alt.Tooltip("recall:Q", format=".3f"),
        alt.Tooltip("f1:Q", format=".3f"),
        "positive_rationales:Q",
    ],
)
topic_rule_labels = topic_rule_bars.mark_text(
    align="left", baseline="middle", dx=4, color="#111111"
).encode(text=alt.Text("f1:Q", format=".3f"))

topic_rule_robustness_chart = (topic_rule_bars + topic_rule_labels).properties(
    width=540,
    height=370,
    title=alt.TitleParams(
        "Topic-Informed Signals Retain Uneven Coherence After Trigger Masking",
        subtitle=[
            "Multilabel TF-IDF classification with five folds grouped by language code.",
            (
                f"Macro-F1: original {topic_rule_macro.loc['Original rationales', 'f1']:.3f}; "
                f"trigger-masked {topic_rule_macro.loc['Trigger-masked rationales', 'f1']:.3f}."
            ),
            "Labels remain rule-derived, not human-validated.",
        ],
        anchor="start",
    ),
).configure_view(stroke=None).configure_legend(orient="top", direction="horizontal")

save_chart(
    topic_rule_robustness_chart,
    str(FIGURE_DIR / "topic_informed_rule_robustness.png"),
    scale_factor=2.0,
)
display(topic_rule_robustness_chart)


Topic-informed rule classifier performance:


,precision,recall,f1
condition,,,
Original rationales,0.875,0.949,0.910
Trigger-masked rationales,0.646,0.840,0.727


,condition,frame,layer,precision,recall,f1,positive_rationales
0,Original rationales,Interdisciplinary field,Definition of DH,0.875,0.975,0.922,3130
1,Original rationales,Technical-methodological intersection,Definition of DH,0.858,0.977,0.914,2913
2,Original rationales,Human-cultural scope,Definition of DH,0.896,0.982,0.937,2262
3,Original rationales,Knowledge-learning scope,Definition of DH,0.866,0.949,0.906,4097
4,Original rationales,Lexical-compositional strategy,Translation authorization,0.898,0.873,0.885,12300
5,Original rationales,Institutional usage/legitimacy,Translation authorization,0.860,0.939,0.898,5825
6,Trigger-masked rationales,Interdisciplinary field,Definition of DH,0.588,0.802,0.679,3130
7,Trigger-masked rationales,Technical-methodological intersection,Definition of DH,0.624,0.898,0.736,2913
8,Trigger-masked rationales,Human-cultural scope,Definition of DH,0.605,0.867,0.713,2262
9,Trigger-masked rationales,Knowledge-learning scope,Definition of DH,0.581,0.840,0.687,4097


alt.LayerChart(...)

## 7.4 — Original Four-Register Benchmark

I retain the original four candidate registers as a deductive benchmark, not as the framework used downstream. Each was operationalised as a small set of phrasal patterns. A rationale may match multiple registers (definitions overlap in practice; this isn't an exclusive classification).

- **Compositional / literal**: rationale describes DH as a combination of *digital* and *humanities* without further scholarly framing. The default fallback register.
- **Scholarly / methodological**: rationale invokes interdisciplinarity, computational methods, scholarship, academic discipline — recognising DH as a *field with internal methodological commitments*.
- **Human-centered**: rationale appeals to humans, people, culture, or human experience — emphasising the *humanities subject* of DH.
- **Generic knowledge**: rationale glosses humanities as *knowledge*, *education*, *learning* — flattening the scholarly specificity into broader cognitive terms.

In this earlier formulation, the contrast was whether the model treats *humanities* as **a scholarly field** (rich), **a category of human experience** (medium), or **a generic knowledge domain** (thin). The compositional register is the baseline — present in nearly every rationale — so the comparative measure is which *non-compositional* register a model reaches for.

In [10]:
REGISTER_PATTERNS = {
    'compositional_literal': [
        r'\bcombines?\b.*\bdigital\b',
        r'\bdigital\b.*\b(?:and|with|plus)\b.*\bhumanities\b',
    ],
    'scholarly_methodological': [
        r'\binterdisciplinary\b',
        r'\bcomputational\s+method',
        r'\bdigital\s+(?:methods?|tools?)\s+(?:to|in|for)\s+(?:study|stud|research|analy)',
        r'\bscholarly\b.*\bdigital\b',
        r'\bdigital\b.*\bscholarship\b',
        r'\bacademic\s+(?:field|discipline)',
    ],
    'human_centered': [
        r'\b(?:study|exploration|investigation)\s+of\s+human\s+culture',
        r'\bhuman\s+experience\b',
        r'\bpeople\b.*\bdigital\b',
        r'\bhuman\b.*\b(?:digital|computer)\b.*\b(?:cultur|stud|experience)',
    ],
    'knowledge_generic': [
        r'\bdigital\b.*\bknowledge\b',
        r'\bknowledge\b.*\bdigital\b',
        r'\bdigital\s+(?:education|learning|wisdom)',
    ],
}

def matches_register(rationale, register):
    return any(re.search(p, rationale, re.IGNORECASE) for p in REGISTER_PATTERNS[register])

# Apply all four registers
for register in REGISTER_PATTERNS:
    rationale_df[f'reg_{register}'] = rationale_df['rationale'].apply(
        lambda r: matches_register(r, register)
    )

# Service-level summary
register_summary = (
    rationale_df.groupby('service')[[f'reg_{r}' for r in REGISTER_PATTERNS]]
    .mean()
    .mul(100).round(1)
    .rename(columns={f'reg_{r}': r for r in REGISTER_PATTERNS})
)
print('Definitional register prevalence (% of rationales per service):\n')
print(register_summary.to_string())

Definitional register prevalence (% of rationales per service):

          compositional_literal  scholarly_methodological  human_centered  knowledge_generic
service                                                                                     
claude                     65.3                      34.8            15.1               34.5
deepseek                   65.6                      39.9            17.9               19.5
gemini                     69.8                      46.7            23.1               27.4
gemma                      64.4                      22.6             7.0               31.6
llama                      66.7                       7.0             8.7               10.4
mistral                    55.1                       9.4             7.0                3.9
openai                     73.0                      38.4            23.4                8.4
qwen                       56.3                      11.0             2.9                9.4


### Definitional Register Rules and Corpus Examples

The table condenses the four overlapping register rules and pairs each one with an actual rationale from the analyzed corpus. Bold text marks the language that activates the relevant pattern. These examples demonstrate how the rules operate; they do not validate either the rationale's linguistic claims or its accompanying translation.


In [11]:
from great_tables import GT

REGISTER_SERVICE_LABELS = {
    "claude": "Claude",
    "openai": "OpenAI",
    "gemini": "Gemini",
    "deepseek": "DeepSeek",
    "llama": "Llama",
    "gemma": "Gemma",
    "qwen": "Qwen",
    "mistral": "Mistral",
}

register_examples = [
    {
        "register": "Compositional / literal",
        "detects": "Defines the phrase by combining or separately translating its two component terms.",
        "service": "mistral",
        "variant": "minimal",
        "language_code": "tab",
        "language_name": "Tabassaran",
        "bold_phrases": ["Digital", "Humanities"],
    },
    {
        "register": "Scholarly / methodological",
        "detects": "Presents DH as scholarship, an academic field, an interdisciplinary field, or a computational method.",
        "service": "deepseek",
        "variant": "github_searcher",
        "language_code": "xal",
        "language_name": "Kalmyk",
        "bold_phrases": ["interdisciplinary field"],
    },
    {
        "register": "Human-centered",
        "detects": "Frames the humanities through people, culture, society, or human experience.",
        "service": "mistral",
        "variant": "github_searcher",
        "language_code": "mwr",
        "language_name": "Marwari",
        "bold_phrases": ["study of human culture"],
    },
    {
        "register": "Generic knowledge",
        "detects": "Glosses the humanities as broad knowledge, learning, education, or wisdom.",
        "service": "mistral",
        "variant": "minimal",
        "language_code": "lmn",
        "language_name": "Lamani",
        "bold_phrases": ["digital", "knowledge"],
    },
]

def _bold_phrases(text, phrases):
    marked = text
    for phrase in phrases:
        marked = re.sub(
            re.escape(phrase),
            lambda match: f"**{match.group(0)}**",
            marked,
            count=1,
            flags=re.IGNORECASE,
        )
    return marked

register_table_rows = []
for example in register_examples:
    match = rationale_df[
        rationale_df["service"].eq(example["service"])
        & rationale_df["variant"].eq(example["variant"])
        & rationale_df["language_code"].eq(example["language_code"])
    ]
    if len(match) != 1:
        raise ValueError(
            "Expected one rationale for "
            f"{example['service']} / {example['variant']} / {example['language_code']}; "
            f"found {len(match)}."
        )
    rationale = match.iloc[0]["rationale"]
    register_table_rows.append({
        "register": example["register"],
        "detects": example["detects"],
        "corpus_example": _bold_phrases(rationale, example["bold_phrases"]),
        "source": (
            f"{REGISTER_SERVICE_LABELS.get(example['service'], example['service'].title())}; "
            f"{'GitHub searcher' if example['variant'] == 'github_searcher' else example['variant'].title()}; "
            f"{example['language_name']} ({example['language_code']})"
        ),
    })

register_rules_table_df = pd.DataFrame(register_table_rows)
register_rules_table = (
    GT(register_rules_table_df)
    .tab_header(
        title="Definitional Registers Used to Analyze LLM Rationales",
        subtitle=(
            "Actual corpus excerpts illustrate the overlapping lexical rules; "
            "bold text marks representative trigger language."
        ),
    )
    .cols_label(
        register="Register",
        detects="What the rule detects",
        corpus_example="Actual rationale excerpt",
        source="Source",
    )
    .fmt_markdown(columns="corpus_example")
    .cols_width(
        register="175px",
        detects="310px",
        corpus_example="520px",
        source="165px",
    )
    .tab_options(
        column_labels_font_weight="bold",
        column_labels_vlines_style="solid",
        column_labels_vlines_width="1px",
        column_labels_vlines_color="#A9A9A9",
        table_body_vlines_style="solid",
        table_body_vlines_width="1px",
        table_body_vlines_color="#D0D0D0",
        table_body_hlines_style="solid",
        table_body_hlines_width="1px",
        table_body_hlines_color="#D9D9D9",
    )
    .tab_source_note(
        source_note=(
            "Source: disagreement_explorer_data.csv after translation/rationale pairing and Tier 2 exclusions. "
            "Registers are non-exclusive lexical signals, not semantic validation."
        )
    )
)

display(register_rules_table)
register_rules_table_path = FIGURE_DIR / "definitional_register_rules_examples_table.png"
register_rules_table.gtsave(str(register_rules_table_path))
print("Saved article table figure.")


GT(_tbl_data=                     register  \
0     Compositional / literal   
1  Scholarly / methodological   
2              Human-centered   
3           Generic knowledge   

                                             detects  \
0  Defines the phrase by combining or separately ...   
1  Presents DH as scholarship, an academic field,...   
2  Frames the humanities through people, culture,...   
3  Glosses the humanities as broad knowledge, lea...   

                                      corpus_example  \
0  Translation of individual words '**Digital**' ...   
1  This translation combines 'Цифрин' (digital) w...   
2  Digital Humanities in English translates to 'O...   
3  Using approximations of '**digital**' (compute...   

                                    source  
0       Mistral; Minimal; Tabassaran (tab)  
1  DeepSeek; GitHub searcher; Kalmyk (xal)  
2  Mistral; GitHub searcher; Marwari (mwr)  
3           Mistral; Minimal; Lamani (lmn)  , _body=<great_tables._gt_data.Body object at 0x14af6f8a0>, _boxhead=Boxhead([ColInfo(var='register', type=<ColInfoTypeEnum.default: 1>, column_label='Register', column_align='left', column_width='175px'), ColInfo(var='detects', type=<ColInfoTypeEnum.default: 1>, column_label='What the rule detects', column_align='left', column_width='310px'), ColInfo(var='corpus_example', type=<ColInfoTypeEnum.default: 1>, column_label='Actual rationale excerpt', column_align='left', column_width='520px'), ColInfo(var='source', type=<ColInfoTypeEnum.default: 1>, column_label='Source', column_align='left', column_width='165px')]), _stub=<great_tables._gt_data.Stub object at 0x14bf3bed0>, _spanners=Spanners([]), _heading=Heading(title='Definitional Registers Used to Analyze LLM Rationales', subtitle='Actual corpus excerpts illustrate the overlapping lexical rules; bold text marks representative trigger language.', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x14bfb4410>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x14a18e650>, _source_notes=['Source: disagreement_explorer_data.csv after translation/rationale pairing and Tier 2 exclusions. Registers are non-exclusive lexical signals, not semantic validation.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x14bfb4550>, _formats=[<great_tables._gt_data.FormatInfo object at 0x14bfb47d0>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='tab

Saved article table figure: ../public_docs/papers/translation-to-transmogrification/figures/analysis/definitional_register_rules_examples_table.png


### Register Overlap

Because the four registers are non-exclusive, I first examine how frequently they co-occur. Each row conditions on rationales matching that register; cells show the percentage that also match the column register. The matrix is therefore directional rather than symmetric. Its purpose is to show whether the rules describe isolated categories or overlapping layers of definition.


In [12]:
REGISTER_DISPLAY = {
    "compositional_literal": "Compositional / literal",
    "scholarly_methodological": "Scholarly / methodological",
    "human_centered": "Human-centered",
    "knowledge_generic": "Generic knowledge",
}
register_order = list(REGISTER_PATTERNS)
register_display_order = [REGISTER_DISPLAY[register] for register in register_order]

overlap_rows = []
for row_register in register_order:
    row_signal = rationale_df[f"reg_{row_register}"]
    row_total = int(row_signal.sum())
    for column_register in register_order:
        column_signal = rationale_df[f"reg_{column_register}"]
        overlap_count = int((row_signal & column_signal).sum())
        overlap_rows.append({
            "row_register": REGISTER_DISPLAY[row_register],
            "column_register": REGISTER_DISPLAY[column_register],
            "row_total": row_total,
            "overlap_count": overlap_count,
            "pct_of_row": overlap_count / row_total * 100 if row_total else 0,
        })

register_overlap_df = pd.DataFrame(overlap_rows)
overlap_base = alt.Chart(register_overlap_df).encode(
    x=alt.X(
        "column_register:N",
        sort=register_display_order,
        title="Also matches",
        axis=alt.Axis(labelAngle=-25),
    ),
    y=alt.Y(
        "row_register:N",
        sort=register_display_order,
        title="Among rationales matching",
    ),
    tooltip=[
        alt.Tooltip("row_register:N", title="Conditioned register"),
        alt.Tooltip("column_register:N", title="Also matches"),
        alt.Tooltip("row_total:Q", title="Rationales in row"),
        alt.Tooltip("overlap_count:Q", title="Matching both"),
        alt.Tooltip("pct_of_row:Q", title="% of row", format=".1f"),
    ],
)
overlap_rect = overlap_base.mark_rect(stroke="white", strokeWidth=2).encode(
    color=alt.Color(
        "pct_of_row:Q",
        scale=alt.Scale(scheme="teals", domain=[0, 100]),
        title="% of row register",
    )
)
overlap_text = overlap_base.transform_calculate(
    overlap_label="format(datum.pct_of_row, '.0f') + '%'"
).mark_text(fontSize=12).encode(
    text="overlap_label:N",
    color=alt.condition(alt.datum.pct_of_row > 55, alt.value("white"), alt.value("black")),
)
register_overlap_chart = (overlap_rect + overlap_text).properties(
    width=430,
    height=300,
    title=alt.TitleParams(
        "Overlap Among Definitional Register Signals",
        subtitle="Cells show the percentage of each row register that also matches the column register.",
        anchor="start",
    ),
).configure_view(stroke=None)

save_chart(
    register_overlap_chart,
    str(FIGURE_DIR / "definitional_register_overlap.png"),
    scale_factor=2.0,
)
display(register_overlap_chart)


alt.LayerChart(...)

### Trigger-Masked Register Classification

A classifier trained to reproduce labels created by lexical rules is not an independent validation: with the trigger words present, the task is partly circular. I use classification here as a narrower coherence test. A multilabel TF-IDF logistic-regression classifier predicts all four overlapping registers under five-fold cross-validation grouped by language code. I then repeat the task after deleting an intentionally broad list of words and phrases used by the register rules. Grouping by language keeps rationales for one language out of both training and test folds.

Performance after masking asks whether the labels remain associated with surrounding rhetorical patterns once their most direct lexical evidence is unavailable. It does **not** establish that the registers are semantically correct, exhaustive, or equivalent to human annotation.


In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_fscore_support
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline

REGISTER_TRIGGER_MASK = re.compile(
    r"\b(?:digital humanities|digital|humanities|combines?|combining|interdisciplinary|"
    r"computational|methods?|tools?|study|studies|research|analysis|analyzing|scholarly|"
    r"scholarship|academic|field|discipline|humanistic|humanist|liberal arts|cultural studies|"
    r"literary studies|historical studies|human culture|human experience|people|human|computer|"
    r"technology|technologies|technological|electronic|information technology|knowledge|education|"
    r"learning|wisdom|culture|society|history)\b",
    re.IGNORECASE,
)

def mask_register_triggers(text):
    return re.sub(r"\s+", " ", REGISTER_TRIGGER_MASK.sub(" ", text)).strip()

register_targets = rationale_df[
    [f"reg_{register}" for register in register_order]
].astype(int).to_numpy()
register_groups = rationale_df["language_code"].to_numpy()
register_cv = GroupKFold(n_splits=5)

classifier_conditions = {
    "Original rationales": rationale_df["rationale"].tolist(),
    "Trigger-masked rationales": rationale_df["rationale"].map(mask_register_triggers).tolist(),
}

register_metric_rows = []
for condition, texts in classifier_conditions.items():
    estimator = Pipeline([
        (
            "tfidf",
            TfidfVectorizer(
                ngram_range=(1, 2),
                min_df=5,
                max_features=50_000,
                sublinear_tf=True,
            ),
        ),
        (
            "classifier",
            OneVsRestClassifier(
                LogisticRegression(
                    max_iter=2_000,
                    class_weight="balanced",
                    solver="liblinear",
                )
            ),
        ),
    ])
    predictions = cross_val_predict(
        estimator,
        texts,
        register_targets,
        groups=register_groups,
        cv=register_cv,
        method="predict",
        n_jobs=-1,
    )
    precision, recall, f1, support = precision_recall_fscore_support(
        register_targets,
        predictions,
        average=None,
        zero_division=0,
    )
    for register, p, r, f, n in zip(register_order, precision, recall, f1, support):
        register_metric_rows.append({
            "condition": condition,
            "register": REGISTER_DISPLAY[register],
            "precision": p,
            "recall": r,
            "f1": f,
            "positive_rationales": int(n),
        })

register_classifier_metrics = pd.DataFrame(register_metric_rows)
macro_metrics = (
    register_classifier_metrics
    .groupby("condition")[["precision", "recall", "f1"]]
    .mean()
    .round(3)
)
print("Macro-averaged grouped out-of-fold performance:")
display(macro_metrics)
print("Per-register grouped out-of-fold performance:")
display(register_classifier_metrics.round(3))

condition_order = ["Original rationales", "Trigger-masked rationales"]
robustness_bars = alt.Chart(register_classifier_metrics).mark_bar().encode(
    y=alt.Y("register:N", sort=register_display_order, title=None),
    yOffset=alt.YOffset("condition:N", sort=condition_order),
    x=alt.X("f1:Q", title="Out-of-fold F1", scale=alt.Scale(domain=[0, 1])),
    color=alt.Color(
        "condition:N",
        sort=condition_order,
        title=None,
        scale=alt.Scale(domain=condition_order, range=["#2A9D8F", "#E76F51"]),
    ),
    tooltip=[
        alt.Tooltip("register:N", title="Register"),
        alt.Tooltip("condition:N", title="Input"),
        alt.Tooltip("precision:Q", format=".3f", title="Precision"),
        alt.Tooltip("recall:Q", format=".3f", title="Recall"),
        alt.Tooltip("f1:Q", format=".3f", title="F1"),
        alt.Tooltip("positive_rationales:Q", title="Positive rationales"),
    ],
)
robustness_labels = robustness_bars.mark_text(
    align="left", baseline="middle", dx=4, color="#111111"
).encode(text=alt.Text("f1:Q", format=".3f"))

register_robustness_chart = (robustness_bars + robustness_labels).properties(
    width=520,
    height=285,
    title=alt.TitleParams(
        "Definitional Registers Remain Predictable After Trigger Masking",
        subtitle=[
            "Multilabel TF-IDF classification with five folds grouped by language code.",
            (
                f"Macro-F1: original {macro_metrics.loc['Original rationales', 'f1']:.3f}; "
                f"trigger-masked {macro_metrics.loc['Trigger-masked rationales', 'f1']:.3f}."
            ),
            "Labels remain rule-derived, not human-validated.",
        ],
        anchor="start",
    ),
).configure_view(stroke=None).configure_legend(
    orient="top",
    direction="horizontal",
)

save_chart(
    register_robustness_chart,
    str(FIGURE_DIR / "definitional_register_trigger_mask_robustness.png"),
    scale_factor=2.0,
)
display(register_robustness_chart)


Macro-averaged grouped out-of-fold performance:


,precision,recall,f1
condition,,,
Original rationales,0.835,0.924,0.875
Trigger-masked rationales,0.631,0.815,0.705


Per-register grouped out-of-fold performance:


,condition,register,precision,recall,f1,positive_rationales
0,Original rationales,Compositional / literal,0.890,0.863,0.876,13321
1,Original rationales,Scholarly / methodological,0.862,0.946,0.902,5433
2,Original rationales,Human-centered,0.718,0.917,0.805,2709
3,Original rationales,Generic knowledge,0.870,0.971,0.918,3770
4,Trigger-masked rationales,Compositional / literal,0.832,0.824,0.828,13321
5,Trigger-masked rationales,Scholarly / methodological,0.630,0.794,0.703,5433
6,Trigger-masked rationales,Human-centered,0.507,0.799,0.621,2709
7,Trigger-masked rationales,Generic knowledge,0.556,0.844,0.670,3770


alt.LayerChart(...)

In [14]:
# Heatmap — service × register, ordered by frontier-then-small
svc_order = sorted(rationale_df['service'].unique(),
                   key=lambda s: (s not in FRONTIER, s))
reg_order = ['compositional_literal', 'scholarly_methodological', 'human_centered', 'knowledge_generic']
service_labels = {
    'claude': 'Claude', 'openai': 'OpenAI', 'gemini': 'Gemini', 'deepseek': 'DeepSeek',
    'llama': 'Llama', 'gemma': 'Gemma', 'qwen': 'Qwen', 'mistral': 'Mistral',
}
register_labels = {
    'compositional_literal': 'Compositional / literal',
    'scholarly_methodological': 'Scholarly / methodological',
    'human_centered': 'Human-centered',
    'knowledge_generic': 'Generic knowledge',
}

long = (register_summary.reset_index()
        .melt(id_vars='service', var_name='register', value_name='pct'))
long['model'] = long['service'].map(service_labels)
long['register_label'] = long['register'].map(register_labels)
svc_display_order = [service_labels[service] for service in svc_order]
reg_display_order = [register_labels[register] for register in reg_order]

heat = alt.Chart(long).mark_rect(stroke='white', strokeWidth=2).encode(
    x=alt.X('register_label:N', sort=reg_display_order, title=None,
            axis=alt.Axis(labelAngle=-30)),
    y=alt.Y('model:N', sort=svc_display_order, title=None),
    color=alt.Color('pct:Q', scale=alt.Scale(scheme='blues', domain=[0, 100]),
                    title='% of rationales'),
    tooltip=['model:N','register_label:N','pct:Q'],
).properties(width=380, height=240,
             title=alt.Title('Definitional register × service',
                             subtitle='Registers overlap; each cell is the share of a service’s rationales matching that signal.'))
txt = heat.mark_text(fontSize=10).encode(
    text=alt.Text('pct:Q', format='.0f'),
    color=alt.condition('datum.pct > 50', alt.value('white'), alt.value('black')),
)
register_chart = heat + txt
save_chart(register_chart, str(FIGURE_DIR / 'definitional_register_by_service.png'))
register_chart

alt.LayerChart(...)

## 7.5 — Component-level: digital vs humanities

The user hypothesis (and a common scholarly observation): in *Digital Humanities*, the **humanities** side is harder to translate than the **digital** side. "Digital" has settled into a stable cross-linguistic concept (computers, technology, electronic data); "humanities" is contested even within English-language academia (scholarly field? humanities-as-people? humanities-as-culture?).

I operationalize this by asking whether a rationale contains an explicit **gloss signal** for either component. A digital signal uses vocabulary such as *technology*, *computational*, *electronic*, or *computer-based*. A humanities signal uses vocabulary such as *scholarship*, *humanistic*, *liberal arts*, *interdisciplinary*, *cultural studies*, or more generic descriptions of humanities fields, knowledge, culture, and experience. I then compare the share of each service's rationales matching the two signal sets.

**How the code determines a gloss:**

- Matching is case-insensitive and limited to the English-register rationales. The regex word boundaries reduce substring matches but do not perform syntactic or semantic parsing.
- `DIGITAL_GLOSS` is intentionally broad. A rationale matches if it invokes computers, computation, electronics, information technology, or technological tools/methods. Because several patterns can match anywhere in the rationale, this means the model uses a technical frame; it does not prove that the relevant phrase directly defines the word *digital*.
- Humanities evidence is split into two layers. `HUMANITIES_GLOSS_SCHOLARLY` looks for field-specific language such as *scholarship*, *liberal arts*, *cultural studies*, or *academic discipline*. `HUMANITIES_GLOSS_GENERIC` captures broader language about humanities fields, knowledge, culture, society, history, or human experience.
- `has_humanities_gloss_any` is the logical OR of those scholarly and generic layers. I use this combined measure in the digital-versus-humanities chart so that humanities is not penalized simply because its explanations are conceptually more varied. The component table retains both layers separately.
- The digital and humanities indicators are independent. One rationale can match both, either, or neither. Percentages therefore describe prevalence among all retained rationales and should not be added together.
- The asymmetry statistic is `digital percentage - humanities-any percentage`. A positive value means the service invokes digital vocabulary more often; a negative value means it invokes humanities vocabulary more often. It measures differential explicitness under these pattern lists, not which component is objectively harder to translate.
- Absence of a match is not evidence that the model lacks the concept. Paraphrases outside the closed pattern lists become false negatives, while incidental mentions, negation, or unrelated uses can become false positives. These are theory-laden screening rules for aggregate comparison, not validated semantic annotations.

In [15]:
DIGITAL_GLOSS = [
    r'\bdigital\s+(?:technology|technologies|tools?|methods?|techniques?)\b',
    r'\bcomput(?:er|ational|ing)\b',
    r'\belectronic\b',
    r'\binformation\s+technology\b',
    r'\btechnolog(?:y|ies|ical)\b',
]

HUMANITIES_GLOSS_SCHOLARLY = [
    r'\b(?:liberal\s+arts|scholarly|scholarship|humanistic|humanist)\b',
    r'\b(?:cultural\s+studies|literary\s+studies|historical\s+studies)\b',
    r'\b(?:academic\s+field|academic\s+discipline)\b',
    r'\binterdisciplinary\s+(?:field|study|research|approach)\b',
]

HUMANITIES_GLOSS_GENERIC = [
    r'\bhumanities\s+(?:fields?|subjects?|disciplines?|areas?)\b',
    r'\bhuman(?:istic|ities)\s+(?:knowledge|topics|matters)\b',
    r'\bhuman\s+(?:culture|experience|society|history)\b',
]

def _any(rat, pats):
    return any(re.search(p, rat, re.IGNORECASE) for p in pats)

rationale_df['has_digital_gloss']            = rationale_df['rationale'].apply(lambda r: _any(r, DIGITAL_GLOSS))
rationale_df['has_humanities_gloss_scholar'] = rationale_df['rationale'].apply(lambda r: _any(r, HUMANITIES_GLOSS_SCHOLARLY))
rationale_df['has_humanities_gloss_generic'] = rationale_df['rationale'].apply(lambda r: _any(r, HUMANITIES_GLOSS_GENERIC))
rationale_df['has_humanities_gloss_any']     = rationale_df['has_humanities_gloss_scholar'] | rationale_df['has_humanities_gloss_generic']

component_summary = (
    rationale_df.groupby('service')[['has_digital_gloss',
                                      'has_humanities_gloss_scholar',
                                      'has_humanities_gloss_generic',
                                      'has_humanities_gloss_any']]
    .mean().mul(100).round(1)
)
print('Component-gloss signal prevalence (% of rationales):\n')
print(component_summary.to_string())
print('\nDigital-vs-humanities-any asymmetry (digital minus humanities, percentage points):')
print((component_summary['has_digital_gloss'] - component_summary['has_humanities_gloss_any']).round(1).to_string())

Component-gloss signal prevalence (% of rationales):

          has_digital_gloss  has_humanities_gloss_scholar  has_humanities_gloss_generic  has_humanities_gloss_any
service                                                                                                          
claude                 37.5                          40.3                          12.7                      46.5
deepseek               39.5                          38.1                          14.6                      45.5
gemini                 42.8                          44.1                          27.2                      57.1
gemma                  48.8                          16.2                          18.3                      31.2
llama                  34.7                          11.0                          14.6                      24.3
mistral                19.6                           7.1                           6.1                      12.5
openai                 56.3       

In [16]:
# Side-by-side bars: digital vs humanities gloss per service
comp_long = (component_summary[['has_digital_gloss', 'has_humanities_gloss_any']]
             .reset_index()
             .melt(id_vars='service', var_name='component', value_name='pct'))
comp_long['component'] = comp_long['component'].map({
    'has_digital_gloss': 'digital',
    'has_humanities_gloss_any': 'humanities (any gloss)',
})
comp_long['model'] = comp_long['service'].map(service_labels)

component_chart = alt.Chart(comp_long).mark_bar().encode(
    x=alt.X('pct:Q', title='% of rationales matching gloss signal'),
    y=alt.Y('model:N', sort=svc_display_order, title=None),
    color=alt.Color('component:N', title='Component',
                    scale=alt.Scale(domain=['digital', 'humanities (any gloss)'],
                                    range=['#1976d2', '#c0680c'])),
    yOffset='component:N',
    tooltip=['model:N','component:N','pct:Q'],
).properties(width=420, height=300,
             title=alt.Title('Component-gloss signals across services',
                             subtitle='Regex prevalence; signals indicate explicit vocabulary, not validated definitions.'))
save_chart(component_chart, str(FIGURE_DIR / 'digital_humanities_component_gloss_by_service.png'))
component_chart

alt.Chart(...)

## 7.6 — Prompt Effects on the Six Overlapping Categories

I now use the topic-informed categories, rather than the original four-register benchmark, for the downstream prompt comparison. The heatmaps show the percentage of each service-prompt corpus matching each signal. Because rationales may activate several categories, cells should be compared rather than summed. The first four panels describe DH; the last two show how translations are authorized.


In [17]:
topic_frame_variant = (
    rationale_df.groupby(["service", "variant"])[topic_signal_cols]
    .mean()
    .mul(100)
    .reset_index()
    .melt(id_vars=["service", "variant"], var_name="frame", value_name="pct")
)
topic_frame_variant["frame"] = topic_frame_variant["frame"].str.replace(
    "topic_frame_", "", regex=False
)
topic_frame_variant["frame_label"] = topic_frame_variant["frame"].map(TOPIC_FRAME_LABELS)
topic_frame_variant["layer"] = topic_frame_variant["frame"].map(TOPIC_FRAME_LAYERS)
topic_frame_variant["model"] = topic_frame_variant["service"].map(service_labels)
topic_frame_variant["prompt"] = topic_frame_variant["variant"].map({
    "minimal": "Minimal", "github_searcher": "GitHub searcher", "judge": "Judge"
})
topic_frame_variant["service_prompt"] = (
    topic_frame_variant["model"] + " · " + topic_frame_variant["prompt"]
)
service_prompt_order = [
    f"{service_labels[service]} · {prompt}"
    for service in LLM_SERVICES
    for prompt in ["Minimal", "GitHub searcher", "Judge"]
]

prompt_frame_base = alt.Chart(topic_frame_variant).encode(
    x=alt.X("prompt:N", sort=["Minimal", "GitHub searcher", "Judge"], title=None),
    y=alt.Y("model:N", sort=topic_model_order, title=None),
    tooltip=[
        "model:N", "prompt:N", "layer:N", "frame_label:N",
        alt.Tooltip("pct:Q", title="% of rationales", format=".1f"),
    ],
)
prompt_frame_rect = prompt_frame_base.mark_rect(stroke="white", strokeWidth=1).encode(
    color=alt.Color("pct:Q", scale=alt.Scale(scheme="viridis", domain=[0, 100]), title="%")
)
prompt_frame_text = prompt_frame_base.mark_text(fontSize=8).encode(
    text=alt.Text("pct:Q", format=".0f"),
    color=alt.condition(alt.datum.pct > 55, alt.value("white"), alt.value("black")),
)
topic_frame_prompt_chart = (prompt_frame_rect + prompt_frame_text).properties(
    width=115,
    height=230,
).facet(
    facet=alt.Facet(
        "frame_label:N", sort=topic_frame_display_order, title=None,
        header=alt.Header(labelFontSize=10, labelLimit=190),
    ),
    columns=3,
    title=alt.TitleParams(
        "Prompt Effects on Topic-Informed Categories by Service",
        subtitle="Percent of retained rationales matching each overlapping signal.",
        anchor="start",
    ),
).resolve_scale(color="shared")
save_chart(
    topic_frame_prompt_chart,
    str(FIGURE_DIR / "topic_informed_frames_by_service_prompt.png"),
    scale_factor=2.0,
)
display(topic_frame_prompt_chart)


alt.FacetChart(...)

## 7.7 — Cross-reference with Language-Level Review Burden

This exploratory comparison now uses all six topic-informed categories. Each rationale inherits its language row's aggregate `flag_count` from `automated_review_signals.csv`; the count summarizes signals across services and is not a quality score for the individual translation attached to that rationale. Differences therefore describe associations with language-level review burden, not translation success or causal effects of a rationale frame.


In [18]:
qf_path = os.path.join(DATA_DIR, "translated_terms", TERM_SLUG, "evaluation", "automated_review_signals.csv")
qf = pd.read_csv(qf_path, dtype=str)
qf["flag_count"] = qf["flag_count"].astype(int)

rationale_q = rationale_df.merge(
    qf[["language_code", "flag_count", "language_family"]],
    on="language_code", how="left",
)
rationale_q["flag_count"] = rationale_q["flag_count"].fillna(0).astype(int)

burden_rows = []
for frame in topic_frame_order:
    col = f"topic_frame_{frame}"
    with_signal = rationale_q.loc[rationale_q[col], "flag_count"]
    without_signal = rationale_q.loc[~rationale_q[col], "flag_count"]
    burden_rows.append({
        "frame": TOPIC_FRAME_LABELS[frame],
        "layer": TOPIC_FRAME_LAYERS[frame],
        "mean_flags_with": with_signal.mean(),
        "mean_flags_without": without_signal.mean(),
        "difference": with_signal.mean() - without_signal.mean(),
        "rationales_with": len(with_signal),
    })
topic_frame_burden = pd.DataFrame(burden_rows)
display(topic_frame_burden.round(3))

burden_points = topic_frame_burden.melt(
    id_vars=["frame", "layer", "difference", "rationales_with"],
    value_vars=["mean_flags_with", "mean_flags_without"],
    var_name="signal_status", value_name="mean_flags",
)
burden_points["signal_status"] = burden_points["signal_status"].map({
    "mean_flags_with": "Matches signal", "mean_flags_without": "Does not match"
})
burden_chart = alt.Chart(burden_points).mark_point(size=95, filled=True).encode(
    y=alt.Y("frame:N", sort=topic_frame_display_order, title=None),
    x=alt.X("mean_flags:Q", title="Mean language-level automated-review flags", scale=alt.Scale(zero=False)),
    color=alt.Color(
        "signal_status:N", title=None,
        scale=alt.Scale(domain=["Matches signal", "Does not match"], range=["#2A9D8F", "#9E9E9E"]),
    ),
    tooltip=["layer:N", "frame:N", "signal_status:N", alt.Tooltip("mean_flags:Q", format=".3f")],
).properties(
    width=560,
    height=310,
    title=alt.TitleParams(
        "Language-Level Review Burden by Topic-Informed Category",
        subtitle="Associational comparison; review flags aggregate across services for each language.",
        anchor="start",
    ),
).configure_view(stroke=None).configure_legend(orient="top", direction="horizontal")
save_chart(
    burden_chart,
    str(FIGURE_DIR / "topic_informed_categories_and_review_flags.png"),
    scale_factor=2.0,
)
display(burden_chart)


,frame,layer,mean_flags_with,mean_flags_without,difference,rationales_with
0,Interdisciplinary field,Definition of DH,3.717,3.994,-0.278,3130
1,Technical-methodological intersection,Definition of DH,3.676,3.998,-0.322,2913
2,Human-cultural scope,Definition of DH,3.910,3.957,-0.047,2262
3,Knowledge-learning scope,Definition of DH,4.152,3.903,0.249,4097
4,Lexical-compositional strategy,Translation authorization,3.909,4.015,-0.106,12300
5,Institutional usage/legitimacy,Translation authorization,3.812,4.007,-0.195,5825


alt.Chart(...)

In [19]:
# Per-service: mean flag count by scholarly-register status
svc_register_flags = (
    rationale_q.groupby(['service', 'reg_scholarly_methodological'])['flag_count']
    .mean().reset_index()
    .rename(columns={'reg_scholarly_methodological': 'has_scholarly'})
)
svc_register_flags['has_scholarly'] = svc_register_flags['has_scholarly'].map({True: 'scholarly', False: 'not scholarly'})
svc_register_flags['model'] = svc_register_flags['service'].map(service_labels)
print('Mean automated flag count by service and scholarly-register status:')
print(svc_register_flags.to_string(index=False))

quality_chart = alt.Chart(svc_register_flags).mark_bar().encode(
    x=alt.X('flag_count:Q', title='Mean language-level flag count'),
    y=alt.Y('model:N', sort=svc_display_order, title=None),
    color=alt.Color('has_scholarly:N', title='Rationale register',
                    scale=alt.Scale(domain=['scholarly', 'not scholarly'],
                                    range=['#1a6b30', '#aaaaaa'])),
    yOffset='has_scholarly:N',
    tooltip=['model:N','has_scholarly:N','flag_count:Q'],
).properties(width=420, height=300,
             title=alt.Title('Language-level review burden by rationale register',
                             subtitle='Flag counts aggregate signals across services; they do not score the individual rationale or translation.'))
save_chart(quality_chart, str(FIGURE_DIR / 'scholarly_rationale_register_and_review_flags.png'))
quality_chart

Mean automated flag count by service and scholarly-register status:
 service has_scholarly  flag_count    model
  claude not scholarly    4.024376   Claude
  claude     scholarly    3.895539   Claude
deepseek not scholarly    4.153262 DeepSeek
deepseek     scholarly    3.695320 DeepSeek
  gemini not scholarly    3.994139   Gemini
  gemini     scholarly    3.913189   Gemini
   gemma not scholarly    3.973827    Gemma
   gemma     scholarly    3.972927    Gemma
   llama not scholarly    3.873407    Llama
   llama     scholarly    3.911765    Llama
 mistral not scholarly    3.956077  Mistral
 mistral     scholarly    3.913223  Mistral
  openai not scholarly    4.233758   OpenAI
  openai     scholarly    3.428864   OpenAI
    qwen not scholarly    4.069906     Qwen
    qwen     scholarly    3.242215     Qwen


alt.Chart(...)

## 7.8 — Limitations and future validation

No single method in this notebook establishes a definitive taxonomy. NMF summarizes recurring vocabulary but requires an interpretive choice about topic count. The BERTopic-style solution is sensitive to density parameters and often recovers service templates or translation procedures alongside disciplinary definitions. The regex signals remain incomplete and biased toward explicit English-language scholarly phrasing, while the trigger-masked classifier predicts labels derived from those same rules rather than independently validated human annotations.

I therefore treat the six topic-informed signals as overlapping reading aids. The agreement among NMF vocabularies, actual examples, and masked classification supports their use for aggregate comparison; disagreement among the methods is evidence against presenting them as mutually exclusive or exhaustive categories. A future human-coded stratified sample could assess precision, recall, and missing frames without introducing another LLM as the authority for what counts as a definition.

**Other natural extensions:**

- **Family-level analysis**: do definitional and authorization signals vary by target language family or by the availability of external linguistic resources?
- **Within-rationale shifts**: does the model start with a scholarly definition and then drift to a literal compositional one as it discusses target-language morphology? Sentence-level tagging would surface this.
- **Component asymmetry within a single sentence**: do the rationales that gloss *digital* also gloss *humanities*, or is the asymmetry happening *within* rationales (not across)?
- **Cross-reference with translation convergence**: do languages with stable translation forms also show more stable definitional or authorization patterns? Such an association would still require care because convergence does not establish linguistic validity.